# Day 13 (Week 2) — Classification 2
**Goal:** Try a second model, tune a parameter, compute precision/recall, compare models, and save results to `compare.csv`.

---

## Step 1 — Imports

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris

Step 2 — Load a Simple Dataset (Binary Classification)

In [2]:
data = load_iris(as_frame=True)
df = data.frame.copy()

# Binary target: class 0 -> 0, classes 1&2 -> 1
df["target"] = (df["target"] != 0).astype(int)

df.head()


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


Step 3 — Prepare X, y + Train/Test Split

In [3]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape


((120, 4), (30, 4))

Step 4 — Model 1 (Baseline): Logistic Regression

In [4]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

print("=== Logistic Regression (Baseline) ===")
print(classification_report(y_test, y_pred_lr))


=== Logistic Regression (Baseline) ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        20

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



Step 5 — Model 2: Decision Tree + Tune max_depth

In [5]:
depths = [1, 2, 3, 4, 5, None]
rows = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    y_pred = dt.predict(X_test)

    rows.append({
        "model": "DecisionTree",
        "param_name": "max_depth",
        "param_value": str(d),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    })

compare_dt = pd.DataFrame(rows)
compare_dt


,model,param_name,param_value,accuracy,precision,recall,f1
0,DecisionTree,max_depth,1,1.0,1.0,1.0,1.0
1,DecisionTree,max_depth,2,1.0,1.0,1.0,1.0
2,DecisionTree,max_depth,3,1.0,1.0,1.0,1.0
3,DecisionTree,max_depth,4,1.0,1.0,1.0,1.0
4,DecisionTree,max_depth,5,1.0,1.0,1.0,1.0
5,DecisionTree,max_depth,None,1.0,1.0,1.0,1.0


Step 6 — Improve Logistic Regression by Tuning C

In [6]:
Cs = [0.01, 0.1, 1, 10, 100]
rows2 = []

for c in Cs:
    lr2 = LogisticRegression(C=c, max_iter=1000)
    lr2.fit(X_train, y_train)
    y_pred = lr2.predict(X_test)

    rows2.append({
        "model": "LogisticRegression",
        "param_name": "C",
        "param_value": str(c),
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    })

compare_lr = pd.DataFrame(rows2)
compare_lr


,model,param_name,param_value,accuracy,precision,recall,f1
0,LogisticRegression,C,0.01,1.0,1.0,1.0,1.0
1,LogisticRegression,C,0.1,1.0,1.0,1.0,1.0
2,LogisticRegression,C,1,1.0,1.0,1.0,1.0
3,LogisticRegression,C,10,1.0,1.0,1.0,1.0
4,LogisticRegression,C,100,1.0,1.0,1.0,1.0


Step 7 — Combine Results + Sort (Pick the Best)

In [7]:
final_compare = pd.concat([compare_dt, compare_lr], ignore_index=True)

final_compare_sorted = final_compare.sort_values(
    by=["f1", "recall", "precision", "accuracy"],
    ascending=False
)

final_compare_sorted


,model,param_name,param_value,accuracy,precision,recall,f1
0,DecisionTree,max_depth,1,1.0,1.0,1.0,1.0
1,DecisionTree,max_depth,2,1.0,1.0,1.0,1.0
2,DecisionTree,max_depth,3,1.0,1.0,1.0,1.0
3,DecisionTree,max_depth,4,1.0,1.0,1.0,1.0
4,DecisionTree,max_depth,5,1.0,1.0,1.0,1.0
5,DecisionTree,max_depth,None,1.0,1.0,1.0,1.0
6,LogisticRegression,C,0.01,1.0,1.0,1.0,1.0
7,LogisticRegression,C,0.1,1.0,1.0,1.0,1.0
8,LogisticRegression,C,1,1.0,1.0,1.0,1.0
9,LogisticRegression,C,10,1.0,1.0,1.0,1.0


Step 8 — Re-train the Best Model + Report + Confusion Matrix

In [8]:
best = final_compare_sorted.iloc[0]
print("BEST ROW:")
print(best)

if best["model"] == "DecisionTree":
    best_depth = None if best["param_value"] == "None" else int(best["param_value"])
    best_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
else:
    best_c = float(best["param_value"])
    best_model = LogisticRegression(C=best_c, max_iter=1000)

best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

print("\n=== BEST MODEL REPORT ===")
print("Model:", best["model"], "|", best["param_name"], "=", best["param_value"])
print(classification_report(y_test, best_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, best_pred))


BEST ROW:
model          DecisionTree
param_name        max_depth
param_value               1
accuracy                1.0
precision               1.0
recall                  1.0
f1                      1.0
Name: 0, dtype: object

=== BEST MODEL REPORT ===
Model: DecisionTree | max_depth = 1
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00        20

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

Confusion Matrix:
[[10  0]
 [ 0 20]]


Step 9 — Save Comparison to compare.csv

In [9]:
final_compare_sorted.to_csv("compare.csv", index=False)
print("Saved: compare.csv")


Saved: compare.csv
